# ISY0101 — Ingeniería de Soluciones con IA
## EP1: Agente RAG — Ripley Chile (versión GitHub Models)
Usa `GITHUB_TOKEN` y `GITHUB_BASE_URL` — sin costo adicional.

## Celda 1 — Instalación

GITHUB_BASE_URL: https://models.inference.ai.azure.com/

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-core langchain-groq faiss-cpu tiktoken sentence-transformers
print('✅ Listo')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.4 MB/s eta 0:00:00
✅ Listo


## Celda 2 — Credenciales GitHub Models

In [ ]:
import os
from getpass import getpass

os.environ['GITHUB_TOKEN']    = getpass('GITHUB_TOKEN: ')
os.environ['GITHUB_BASE_URL'] = 'https://models.inference.ai.azure.com'

print('BASE URL:', os.environ['GITHUB_BASE_URL'])
print('✅ Credenciales guardadas')

GITHUB_TOKEN: ··········
BASE URL: https://models.inference.ai.azure.com
✅ Credenciales guardadas


## Celda 3 — Crear datos simulados

In [ ]:
import os, csv
os.makedirs('data', exist_ok=True)

# CSV reclamos históricos
reclamos = [
    ['id','fecha','tipo','descripcion','resolucion','dias_resolucion'],
    ['001','2024-01-10','Garantia','TV dañado sin golpes en 2 semanas','Cambio de producto aceptado',3],
    ['002','2024-01-12','Despacho','Pedido no llegó en plazo prometido','Reembolso de costo de despacho',2],
    ['003','2024-01-15','Devolucion','Ropa talla incorrecta recibida','Cambio por talla correcta',4],
    ['004','2024-01-18','Garantia','Celular no enciende al día siguiente','Cambio inmediato en tienda',1],
    ['005','2024-01-20','Cobro','Cobro duplicado en tarjeta Ripley','Reversión del cobro en 5 días',5],
    ['006','2024-01-22','Despacho','Producto equivocado enviado','Retiro y reenvío correcto',3],
    ['007','2024-01-25','Garantia','Lavadora con ruido extraño tras 1 mes','Reparación sin costo',7],
    ['008','2024-01-28','Devolucion','Embalaje dañado al recibir','Devolución y reembolso completo',2],
]
with open('data/reclamos_historicos.csv','w',newline='',encoding='utf-8') as f:
    csv.writer(f).writerows(reclamos)

# Políticas de garantía
with open('data/politicas_garantia.txt','w',encoding='utf-8') as f:
    f.write("""POLÍTICAS DE GARANTÍA Y DEVOLUCIÓN — RIPLEY CHILE

1. GARANTÍA LEGAL
Todo producto vendido por Ripley Chile tiene garantía mínima de 3 meses.
Para tecnología (televisores, celulares, computadores) la garantía es de 6 meses.
Conforme a la Ley 19.496 del Consumidor.

2. PROCESO DE GARANTÍA
El cliente debe presentar boleta o comprobante en cualquier tienda Ripley.
Plazo de respuesta: 5 días hábiles desde la recepción del producto.
Si no puede repararse, se procede al cambio o devolución del dinero.

3. DEVOLUCIONES
Plazo: 10 días hábiles desde la compra, embalaje original y sin uso.
No aplica: ropa interior, higiene personal, software con sello roto.

4. DESPACHO
Si el pedido no llega en el plazo prometido, el cliente puede pedir reembolso del despacho.
Si no llega tras 5 días del plazo, puede cancelar el pedido con reembolso total.

5. COBROS INCORRECTOS
Reversión dentro de 5 días hábiles con presentación del comprobante de pago.
""")

# FAQ
with open('data/faq_atencion.txt','w',encoding='utf-8') as f:
    f.write("""PREGUNTAS FRECUENTES — ATENCIÓN AL CLIENTE RIPLEY

P: ¿Cuánto tiempo tengo para hacer una devolución?
R: 10 días hábiles desde la compra, producto en estado original.

P: ¿Qué hago si recibí un producto dañado?
R: Contáctanos dentro de 24 horas. Coordinaremos retiro sin costo y reenvío o reembolso.

P: ¿Mi televisor tiene garantía si la pantalla se dañó sola?
R: Sí, si el daño no es imputable al consumidor. Garantía de 6 meses. Presenta tu boleta en tienda.

P: ¿Puedo pedir devolución del dinero en vez de reparación?
R: Sí. Según la Ley 19.496 puedes elegir entre reparación, cambio o devolución del dinero.

P: ¿Qué pasa si Ripley no soluciona mi reclamo?
R: Puedes ir al SERNAC en www.sernac.cl. Ripley debe responder en 10 días hábiles.
""")

# Ley 19.496
with open('data/ley_19496.txt','w',encoding='utf-8') as f:
    f.write("""LEY 19.496 — PROTECCIÓN DE LOS DERECHOS DE LOS CONSUMIDORES (Extracto)

Artículo 20:
El consumidor podrá optar entre la reparación gratuita del bien o su reposición
o la devolución de la cantidad pagada en los siguientes casos:
a) Cuando el bien no corresponda al tipo, calidad o cantidad convenidas.
b) Cuando el bien presente defectos no visibles al momento de la compra.
El plazo para ejercer estas acciones es de 3 meses desde la fecha de la compra.
Para bienes con garantía voluntaria mayor, rige el plazo de la garantía.

Artículo 23:
Comete infracción el proveedor que, actuando con negligencia, causa menoscabo al
consumidor debido a fallas o deficiencias en la calidad del bien o servicio.

Artículo 58:
El SERNAC velará por el cumplimiento de la presente ley. Podrá recibir reclamos
de consumidores y el proveedor deberá dar respuesta en un plazo de 10 días hábiles.
""")

print('✅ Archivos creados:')
for f in os.listdir('data'): print(f'   - {f}')

✅ Archivos creados:
   - faq_atencion.txt
   - politicas_garantia.txt
   - ley_19496.txt
   - reclamos_historicos.csv


## Celda 4 — Indexación con FAISS

In [ ]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Cargar documentos
documentos = []
for nombre, tipo in [('politicas_garantia.txt','interno'),
                     ('faq_atencion.txt','interno'),
                     ('ley_19496.txt','externo')]:
    docs = TextLoader(f'data/{nombre}', encoding='utf-8').load()
    for d in docs:
        d.metadata['source'] = nombre
        d.metadata['tipo']   = tipo
    documentos.extend(docs)

docs = CSVLoader('data/reclamos_historicos.csv', encoding='utf-8').load()
for d in docs:
    d.metadata['source'] = 'reclamos_historicos.csv'
    d.metadata['tipo']   = 'interno'
documentos.extend(docs)
print(f'Documentos cargados: {len(documentos)}')

# Chunking
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50
).split_documents(documentos)
print(f'Chunks generados: {len(chunks)}')

# Embeddings locales multilingues (gratis, sin API key)
print('Cargando modelo de embeddings local (primera vez ~1 min)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

# Crear y guardar índice FAISS
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local('faiss_index')
print('✅ Índice FAISS creado y guardado')

Documentos cargados: 11
Chunks generados: 15
Cargando modelo de embeddings local (primera vez ~1 min)...


/tmp/ipykernel_5662/1149075121.py:32: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Índice FAISS creado y guardado


In [ ]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Ingresa tu GROQ_API_KEY: ')
print('✅ Groq key guardada')

Ingresa tu GROQ_API_KEY: ··········
✅ Groq key guardada


## Celda 5 — Construir el agente RAG con GPT-4o via GitHub

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import os, json

# Cargar índice
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)
vector_store = FAISS.load_local(
    'faiss_index', embeddings, allow_dangerous_deserialization=True
)
retriever = vector_store.as_retriever(search_kwargs={'k': 4})

# LLM via Groq (gratuito)
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0,
    api_key=os.environ['GROQ_API_KEY']
)

SYSTEM_PROMPT = """Eres un agente especializado en atención de reclamos postventa de Ripley Chile.
Tu función es clasificar reclamos, sugerir respuestas y determinar si corresponde escalar al SERNAC.

Reglas:
1. Basa SIEMPRE tu respuesta en el contexto proporcionado.
2. Si el contexto no tiene info suficiente, indícalo. No inventes datos.
3. Cita la fuente con el formato [Fuente: nombre_archivo].
4. Aplica la Ley 19.496 cuando corresponda.
5. Responde en español formal y empático.

Responde ÚNICAMENTE con JSON válido:
{{
  "clasificacion": "<tipo de reclamo>",
  "respuesta_sugerida": "<texto para el cliente>",
  "escalar_sernac": true o false,
  "justificacion_escalamiento": "<razón o vacío>",
  "fuentes_usadas": ["<fuente1>", "<fuente2>"]
}}"""

USER_PROMPT = """CONTEXTO RECUPERADO:
{contexto}

---
RECLAMO DEL CLIENTE:
{reclamo}

Responde en formato JSON."""

prompt_template = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('user', USER_PROMPT)
])

def formatear_contexto(docs):
    return '\n\n'.join(
        f"--- Fragmento {i} [Fuente: {d.metadata.get('source','?')} | "
        f"Tipo: {d.metadata.get('tipo','')}] ---\n{d.page_content}"
        for i, d in enumerate(docs, 1)
    )

def procesar_reclamo(texto):
    docs_rel  = retriever.invoke(texto)
    contexto  = formatear_contexto(docs_rel)
    respuesta = (prompt_template | llm).invoke({'contexto': contexto, 'reclamo': texto})
    raw = respuesta.content.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    return json.loads(raw.strip()), docs_rel

print('✅ Agente RAG listo')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Agente RAG listo


In [ ]:
import os
print("BASE URL en memoria:", repr(os.environ.get('GITHUB_BASE_URL')))

os.environ['GITHUB_BASE_URL'] = 'https://models.inference.ai.azure.com'

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0,
    openai_api_key=os.environ['GITHUB_TOKEN'],
    openai_api_base='https://models.inference.ai.azure.com'
)
print("✅ LLM recreado con URL correcto")

BASE URL en memoria: 'https://models.inference.ai.azure.com'
✅ LLM recreado con URL correcto


## Celda 6 — Prueba 1: Garantía de televisor

In [ ]:
reclamo = """
Compré un televisor Samsung hace 3 semanas y se dañó la pantalla sin ningún golpe.
Fui a la tienda Ripley y me dijeron que no tiene garantía porque es daño del cliente.
Exijo cambio del producto o devolución del dinero.
"""
print('RECLAMO:', reclamo.strip())
print('=' * 60)
resultado, docs = procesar_reclamo(reclamo)
print('Fragmentos recuperados:')
for i, d in enumerate(docs, 1):
    print(f'  [{i}] {d.metadata["source"]} — {d.page_content[:70]}...')
print(f'\nClasificación  : {resultado["clasificacion"]}')
print(f'Escalar SERNAC : {resultado["escalar_sernac"]}')
print(f'Fuentes usadas : {", ".join(resultado["fuentes_usadas"])}')
print(f'\nRespuesta:\n{resultado["respuesta_sugerida"]}')

RECLAMO: Compré un televisor Samsung hace 3 semanas y se dañó la pantalla sin ningún golpe.
Fui a la tienda Ripley y me dijeron que no tiene garantía porque es daño del cliente.
Exijo cambio del producto o devolución del dinero.
Fragmentos recuperados:
  [1] faq_atencion.txt — PREGUNTAS FRECUENTES — ATENCIÓN AL CLIENTE RIPLEY

P: ¿Cuánto tiempo t...
  [2] ley_19496.txt — Artículo 20:
El consumidor podrá optar entre la reparación gratuita de...
  [3] politicas_garantia.txt — POLÍTICAS DE GARANTÍA Y DEVOLUCIÓN — RIPLEY CHILE

1. GARANTÍA LEGAL
T...
  [4] politicas_garantia.txt — 3. DEVOLUCIONES
Plazo: 10 días hábiles desde la compra, embalaje origi...

Clasificación  : Reclamo por garantía
Escalar SERNAC : False
Fuentes usadas : politicas_garantia.txt, ley_19496.txt, faq_atencion.txt

Respuesta:
Estimado cliente, lamentamos el inconveniente con su televisor Samsung. Según nuestra política de garantía, si el daño no es imputable al consumidor, el producto tiene garantía de 6 meses. Le ped

## Celda 7 — Prueba 2: Despacho con retraso

In [ ]:
reclamo = """
Hice una compra online hace 10 días y el plazo prometido era 3 días hábiles.
El pedido nunca llegó y al llamar me dicen que está en camino pero sin fecha.
Quiero la devolución inmediata del dinero.
"""
print('RECLAMO:', reclamo.strip())
print('=' * 60)
resultado, docs = procesar_reclamo(reclamo)
print('Fragmentos recuperados:')
for i, d in enumerate(docs, 1):
    print(f'  [{i}] {d.metadata["source"]} — {d.page_content[:70]}...')
print(f'\nClasificación  : {resultado["clasificacion"]}')
print(f'Escalar SERNAC : {resultado["escalar_sernac"]}')
print(f'Fuentes usadas : {", ".join(resultado["fuentes_usadas"])}')
print(f'\nRespuesta:\n{resultado["respuesta_sugerida"]}')

RECLAMO: Hice una compra online hace 10 días y el plazo prometido era 3 días hábiles.
El pedido nunca llegó y al llamar me dicen que está en camino pero sin fecha.
Quiero la devolución inmediata del dinero.
Fragmentos recuperados:
  [1] politicas_garantia.txt — 3. DEVOLUCIONES
Plazo: 10 días hábiles desde la compra, embalaje origi...
  [2] faq_atencion.txt — PREGUNTAS FRECUENTES — ATENCIÓN AL CLIENTE RIPLEY

P: ¿Cuánto tiempo t...
  [3] ley_19496.txt — Artículo 20:
El consumidor podrá optar entre la reparación gratuita de...
  [4] reclamos_historicos.csv — id: 002
fecha: 2024-01-12
tipo: Despacho
descripcion: Pedido no llegó ...

Clasificación  : Despacho
Escalar SERNAC : False
Fuentes usadas : politicas_garantia.txt, faq_atencion.txt, ley_19496.txt

Respuesta:
Estimado cliente, lamentamos el inconveniente con su pedido. Según nuestra política, si el pedido no llega en el plazo prometido, puede pedir reembolso del despacho. Además, como ha pasado más de 5 días desde el plazo prometido,

## Celda 8 — Prueba 3: Caso que escala al SERNAC

In [ ]:
reclamo = """
Llevo 3 semanas intentando que Ripley cambie un refrigerador defectuoso comprado hace 2 meses.
Cada vez me dicen que lo evaluarán pero nadie viene ni dan solución.
Ya perdí los alimentos por la falla. Exijo solución o voy al SERNAC.
"""
print('RECLAMO:', reclamo.strip())
print('=' * 60)
resultado, docs = procesar_reclamo(reclamo)
print('Fragmentos recuperados:')
for i, d in enumerate(docs, 1):
    print(f'  [{i}] {d.metadata["source"]} — {d.page_content[:70]}...')
print(f'\nClasificación        : {resultado["clasificacion"]}')
print(f'Escalar SERNAC       : {resultado["escalar_sernac"]}')
if resultado.get('justificacion_escalamiento'):
    print(f'Justificación escal. : {resultado["justificacion_escalamiento"]}')
print(f'Fuentes usadas       : {", ".join(resultado["fuentes_usadas"])}')
print(f'\nRespuesta:\n{resultado["respuesta_sugerida"]}')

RECLAMO: Llevo 3 semanas intentando que Ripley cambie un refrigerador defectuoso comprado hace 2 meses.
Cada vez me dicen que lo evaluarán pero nadie viene ni dan solución.
Ya perdí los alimentos por la falla. Exijo solución o voy al SERNAC.
Fragmentos recuperados:
  [1] faq_atencion.txt — P: ¿Puedo pedir devolución del dinero en vez de reparación?
R: Sí. Seg...
  [2] politicas_garantia.txt — POLÍTICAS DE GARANTÍA Y DEVOLUCIÓN — RIPLEY CHILE

1. GARANTÍA LEGAL
T...
  [3] politicas_garantia.txt — 3. DEVOLUCIONES
Plazo: 10 días hábiles desde la compra, embalaje origi...
  [4] reclamos_historicos.csv — id: 007
fecha: 2024-01-25
tipo: Garantia
descripcion: Lavadora con rui...

Clasificación        : Garantia
Escalar SERNAC       : True
Justificación escal. : El cliente ha esperado 3 semanas sin recibir una solución, lo que supera el plazo de respuesta establecido en nuestra política de garantía y en la Ley 19.496.
Fuentes usadas       : politicas_garantia.txt, faq_atencion.txt

Respuesta:


## Celda 9 — Tu propio reclamo

In [ ]:
mi_reclamo = """
Compré unos zapatos online y me llegaron de una talla diferente a la que pedí.
Quiero cambiarlos pero no sé si debo pagar el despacho del cambio.
"""
print('RECLAMO:', mi_reclamo.strip())
print('=' * 60)
resultado, docs = procesar_reclamo(mi_reclamo)
print('Fragmentos recuperados:')
for i, d in enumerate(docs, 1):
    print(f'  [{i}] {d.metadata["source"]} — {d.page_content[:70]}...')
print(f'\nClasificación  : {resultado["clasificacion"]}')
print(f'Escalar SERNAC : {resultado["escalar_sernac"]}')
print(f'Fuentes usadas : {", ".join(resultado["fuentes_usadas"])}')
print(f'\nRespuesta:\n{resultado["respuesta_sugerida"]}')

RECLAMO: Compré unos zapatos online y me llegaron de una talla diferente a la que pedí.
Quiero cambiarlos pero no sé si debo pagar el despacho del cambio.
Fragmentos recuperados:
  [1] politicas_garantia.txt — 3. DEVOLUCIONES
Plazo: 10 días hábiles desde la compra, embalaje origi...
  [2] ley_19496.txt — Artículo 20:
El consumidor podrá optar entre la reparación gratuita de...
  [3] faq_atencion.txt — P: ¿Puedo pedir devolución del dinero en vez de reparación?
R: Sí. Seg...
  [4] politicas_garantia.txt — POLÍTICAS DE GARANTÍA Y DEVOLUCIÓN — RIPLEY CHILE

1. GARANTÍA LEGAL
T...

Clasificación  : Reclamo por defecto de calidad o cantidad convenida
Escalar SERNAC : False
Fuentes usadas : politicas_garantia.txt, ley_19496.txt, faq_atencion.txt

Respuesta:
Estimado cliente, lamentamos el inconveniente con su compra. Según nuestra política de garantía y la Ley 19.496, usted tiene derecho a cambiar o devolver el producto. En este caso, no es necesario que pague el despacho del cambio, ya que 

## Celda 10 — Análisis de coherencia (IE4)

In [ ]:
print('ANÁLISIS DE COHERENCIA DATOS → RESPUESTA (IE4)')
print('=' * 60)

casos = [
    'Televisor dañado a los 20 días de compra, piden que pague la reparación',
    'Pedido de despacho lleva 8 días de retraso sin respuesta de Ripley',
    'Me cobraron dos veces el mismo producto en mi tarjeta Ripley'
]

for i, r in enumerate(casos, 1):
    resultado, docs = procesar_reclamo(r)
    fuentes_rec  = list(set(d.metadata.get('source','?') for d in docs))
    fuentes_resp = resultado.get('fuentes_usadas', [])
    coherente    = any(f in fuentes_resp for f in fuentes_rec)
    print(f'\nCaso {i}: {r}')
    print(f'  Chunks recuperados  : {fuentes_rec}')
    print(f'  Fuentes en respuesta: {fuentes_resp}')
    print(f'  Coherencia          : {"✅ OK" if coherente else "⚠️ Revisar"}')
    print(f'  Clasificación       : {resultado["clasificacion"]}')
    print(f'  Escalar SERNAC      : {resultado["escalar_sernac"]}')

ANÁLISIS DE COHERENCIA DATOS → RESPUESTA (IE4)

Caso 1: Televisor dañado a los 20 días de compra, piden que pague la reparación
  Chunks recuperados  : ['politicas_garantia.txt', 'faq_atencion.txt', 'ley_19496.txt']
  Fuentes en respuesta: ['faq_atencion.txt', 'politicas_garantia.txt', 'ley_19496.txt']
  Coherencia          : ✅ OK
  Clasificación       : Reclamo por producto dañado
  Escalar SERNAC      : False

Caso 2: Pedido de despacho lleva 8 días de retraso sin respuesta de Ripley
  Chunks recuperados  : ['faq_atencion.txt', 'politicas_garantia.txt', 'reclamos_historicos.csv']
  Fuentes en respuesta: ['politicas_garantia.txt', 'reclamos_historicos.csv']
  Coherencia          : ✅ OK
  Clasificación       : Despacho
  Escalar SERNAC      : False

Caso 3: Me cobraron dos veces el mismo producto en mi tarjeta Ripley
  Chunks recuperados  : ['politicas_garantia.txt', 'faq_atencion.txt', 'reclamos_historicos.csv']
  Fuentes en respuesta: ['politicas_garantia.txt', 'reclamos_historicos.c